# 05 — XGBoost Classifier: Chega Growth in Legislative Elections (2024→2025)
Predicts whether Chega's vote share grew between the 2024 and 2025 legislative elections, using sociodemographic and historical electoral features.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import shap
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('tabela_principal_2023_2025_legislativas_v0.csv', sep=';')

In [ ]:
print(df.columns.tolist())

In [ ]:
df["crescimento_chega"] = (df["CH_2025"] - df["CH_2024"]).apply(lambda x: 1 if x > 0 else 0)


In [ ]:
# target
y = df["crescimento_chega"]

# features: remover colunas que não devem entrar
X = df.drop(columns=[
    "Concelho",           # identificador
    "CH_2021",            # votes passados do Chega → não pode entrar
    "CH_2022",            # target já definido
    "crescimento_chega" # target
])

In [ ]:
X_clean = X.copy()
X_clean.columns = [c.replace("[","_").replace("]","_").replace(":","_").replace(" ","_") for c in X_clean.columns]


In [ ]:
loo = LeaveOneOut()
y_true, y_pred = [], []

# 7️⃣ Loop Leave-One-Out
for train_idx, test_idx in loo.split(X_clean):
    X_train, X_test = X_clean.iloc[train_idx], X_clean.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        use_label_encoder=False,
        eval_metric="logloss"
    )

    model.fit(X_train, y_train)
    y_hat = model.predict(X_test)

    y_true.append(y_test.values[0])
    y_pred.append(y_hat[0])

In [ ]:
print("Accuracy LOO:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))

In [ ]:
# Criar DataFrame com feature names e suas importâncias
feature_importance = pd.DataFrame({
    "Feature": X_clean.columns,
    "Importance": model.feature_importances_
})

# Ordenar pela importância (decrescente)
feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

# Mostrar todas as features com importância > 0
print("Features relevantes:")
print(feature_importance[feature_importance["Importance"] > 0])

# Mostrar features com importância zero ou muito baixa
print("\nFeatures pouco relevantes ou irrelevantes:")
print(feature_importance[feature_importance["Importance"] < 0.001])

### Teste com balanceamento

In [ ]:
# -------------------------------
# 1️⃣ Separar features e target
# -------------------------------
X = df.drop(columns=["Concelho", "crescimento_chega"])
X.columns = [c.replace("[","_").replace("]","_").replace(":","_").replace(" ","_").replace("<","_") for c in X.columns]
y = df["crescimento_chega"]

# -------------------------------
# 2️⃣ Calcular scale_pos_weight
# -------------------------------
n_positivos = np.sum(y == 1)
n_negativos = np.sum(y == 0)
scale_pos_weight = n_negativos / n_positivos
print("scale_pos_weight:", scale_pos_weight)

# -------------------------------
# 3️⃣ Inicializar Leave-One-Out
# -------------------------------
loo = LeaveOneOut()
y_true, y_pred = [], []

# -------------------------------
# 4️⃣ Loop LOO
# -------------------------------
for train_idx, test_idx in loo.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        use_label_encoder=False,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight  # balanceamento
    )

    model.fit(X_train, y_train)
    y_hat = model.predict(X_test)

    y_true.append(y_test.values[0])
    y_pred.append(y_hat[0])

# -------------------------------
# 5️⃣ Avaliar resultados
# -------------------------------
print("Accuracy LOO:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred))
